In [1]:
import pandas as pd
from pathlib import Path

files = [
    ("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/it_jobs.xlsx",  "it"),
    ("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/cs_jobs.xlsx",  "cs"),
    ("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/swe_jobs.xlsx", "swe"),
    ("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/ds_jobs.xlsx",  "ds"),
    ("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/pm_jobs.xlsx",  "pm"),
]

def pick_col(df, candidates):
    """Pick the first existing column name from a list."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

dfs = []

for fp, src in files:
    df = pd.read_excel(fp)

    # cố gắng tự nhận diện cột title / description (tùy file có tên cột gì)
    title_col = pick_col(df, ["title", "job_title", "Title", "Job Title", "name", "position", "position_title"])
    desc_col  = pick_col(df, ["cleaned_description", "description", "job_description", "Job Description", "desc", "requirements"])

    if title_col is None or desc_col is None:
        print(f"[WARN] File {fp} thiếu cột title/description. Columns hiện có: {list(df.columns)[:15]} ...")
        continue

    out = pd.DataFrame({
        "title": df[title_col].astype(str).fillna(""),
        "description": df[desc_col].astype(str).fillna(""),
    })

    # thêm vài cột nếu có thì giữ lại
    company_col   = pick_col(df, ["company", "Company", "company_name", "employer"])
    location_col  = pick_col(df, ["location", "Location", "city", "job_location"])
    if company_col is not None:
        out["company"] = df[company_col].astype(str).fillna("")
    else:
        out["company"] = ""

    if location_col is not None:
        out["location"] = df[location_col].astype(str).fillna("")
    else:
        out["location"] = ""

    out["source"] = src
    dfs.append(out)

jobs_merged = pd.concat(dfs, ignore_index=True)

# dọn chút cho sạch: bỏ dòng quá rỗng
jobs_merged["title"] = jobs_merged["title"].str.strip()
jobs_merged["description"] = jobs_merged["description"].str.strip()
jobs_merged = jobs_merged[(jobs_merged["title"] != "") & (jobs_merged["description"] != "")]

# bỏ trùng: title + company + location + description (giảm phẳng)
jobs_merged = jobs_merged.drop_duplicates(subset=["title","company","location","description"])

print("Merged jobs shape:", jobs_merged.shape)
jobs_merged.head(5)


Merged jobs shape: (89277, 5)


,title,description,company,location,source
0,Support Technologist II 2024-02216,**description and functions** ----------------...,State of Wyoming,"Rock Springs, WY",it
1,Computer Technician,***if you would like to apply for this positio...,Medford Township Public Schools,"Medford, NJ",it
2,IT Technician,i need a sped up recording converted to text. ...,Law Office of Ruben and Ruben LLC,nan,it
3,IT Rotational Program - June 2025,donnelley financial solutions (dfin) is a lead...,Donnelley Financial Solutions,United States,it
4,Technical Support Specialist,triple whale is a leader in business intellige...,Triple Whale,"Columbus, OH",it


In [2]:
out_path = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx")
jobs_merged.to_excel(out_path, index=False)
print("Saved:", out_path)


Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx
